# House Prices: Linear Regression, Decision Tree, and Random Forest

Assignment: implement Linear Regression, Decision Tree, and Random Forest on a Kaggle-style dataset; apply multiple models; compare performance.

**Dataset:** `house_prices_dataset.csv` (10,000 rows).

| Column | Role |
|---|---|
| `square_feet` | feature |
| `num_rooms` | feature |
| `age` | feature |
| `distance_to_city(km)` | feature |
| `price` | **target** (continuous) |

`price` is numeric, so the matching models are **regressors**. Random Forest Classifier (RFC) is included as a second task after binning price into Low / Medium / High, because a classifier cannot predict a continuous dollar amount.


## 1. Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from scipy.stats import randint

sns.set_theme(style="whitegrid")

df_hp = pd.read_csv("house_prices_dataset.csv")
df_hp.head()


In [ ]:
print("Shape:", df_hp.shape)
print("\nColumn dtypes:")
print(df_hp.dtypes)
print("\nMissing values:")
print(df_hp.isnull().sum())
df_hp.info()


In [ ]:
df_hp.describe()

## 2. Clean invalid prices

This file contains a small number of **negative prices**, which are not valid sale amounts. They are dropped before modeling so metrics are not pulled down by impossible labels.


In [ ]:
n_negative = (df_hp["price"] < 0).sum()
print(f"Negative price rows: {n_negative}")

df_hp = df_hp[df_hp["price"] >= 0].copy().reset_index(drop=True)
print("Rows after dropping negative prices:", len(df_hp))


## 3. Exploratory plots

Each house is an independent row, so **scatter plots** (not line plots) are the right view of `square_feet` / `age` vs `price`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(ax=axes[0], x="square_feet", y="price", data=df_hp, alpha=0.35)
axes[0].set_title("Price vs Square Feet")

sns.scatterplot(ax=axes[1], x="age", y="price", data=df_hp, alpha=0.35, color="orange")
axes[1].set_title("Price vs Age")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(ax=axes[0], x="num_rooms", y="price", data=df_hp)
axes[0].set_title("Price by Number of Rooms")

sns.scatterplot(
    ax=axes[1],
    x="distance_to_city(km)",
    y="price",
    data=df_hp,
    alpha=0.35,
    color="green",
)
axes[1].set_title("Price vs Distance to City")

plt.tight_layout()
plt.show()


## 4. Feature engineering

- `age_x_distance` is a valid interaction **feature** (built only from predictors).
- `price_per_sqft` uses the **target**, so it is for EDA only and is **not** passed into any model.


In [ ]:
df_hp["price_per_sqft"] = df_hp["price"] / df_hp["square_feet"]
df_hp["age_x_distance"] = df_hp["age"] * df_hp["distance_to_city(km)"]

feature_cols = [
    "square_feet",
    "num_rooms",
    "age",
    "distance_to_city(km)",
    "age_x_distance",
]
target_col = "price"

df_hp[feature_cols + [target_col]].head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(ax=axes[0], x="square_feet", y="price_per_sqft", data=df_hp, alpha=0.35)
axes[0].set_title("Price per Sqft vs Square Feet (EDA only)")

sns.histplot(ax=axes[1], x=df_hp["age_x_distance"], bins=50, kde=True, color="green")
axes[1].set_title("Distribution of Age x Distance")

plt.tight_layout()
plt.show()


In [ ]:
corr_cols = feature_cols + ["price_per_sqft", target_col]
plt.figure(figsize=(9, 7))
sns.heatmap(df_hp[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()


## 5. Train / test split (once)

The same split is reused for every regressor so the comparison table is fair.


In [ ]:
X = df_hp[feature_cols]
y = df_hp[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, r2


def print_regression_metrics(name, y_true, y_pred):
    mse, rmse, r2 = regression_metrics(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"MSE:  {mse:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R2:   {r2:.4f}")
    return mse, rmse, r2


def plot_actual_vs_pred(y_true, y_pred, title, color="C0"):
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.3, color=color)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    plt.plot([lo, hi], [lo, hi], "r--", lw=2)
    plt.xlabel("Actual Price")
    plt.ylabel("Predicted Price")
    plt.title(title)
    plt.show()


## 6. Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

mse_lr, rmse_lr, r2_lr = print_regression_metrics(
    "Linear Regression", y_test, y_pred_lr
)
print("\nCoefficients:")
print(pd.Series(lr_model.coef_, index=feature_cols))
print("Intercept:", lr_model.intercept_)


In [ ]:
plot_actual_vs_pred(y_test, y_pred_lr, "Linear Regression: Actual vs Predicted")


## 7. Decision Tree Regressor

In [ ]:
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

mse_dt, rmse_dt, r2_dt = print_regression_metrics(
    "Decision Tree Regressor (base)", y_test, y_pred_dt
)


In [ ]:
plot_actual_vs_pred(
    y_test, y_pred_dt, "Decision Tree: Actual vs Predicted", color="C1"
)


## 8. Random Forest Regressor

This is the correct Random Forest family for continuous `price`. Classifier metrics (accuracy) would be meaningless on raw prices.


In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

mse_rf, rmse_rf, r2_rf = print_regression_metrics(
    "Random Forest Regressor (base)", y_test, y_pred_rf
)


In [ ]:
plot_actual_vs_pred(
    y_test, y_pred_rf, "Random Forest: Actual vs Predicted", color="green"
)


## 9. Hyperparameter tuning (regressors)

Ordinary Linear Regression has no tree-style hyperparameters to grid-search. Tuning is applied to Decision Tree and Random Forest.

Grid search for Random Forest is kept modest so this notebook can finish in a reasonable time.


In [ ]:
dt_param_grid = {
    "max_depth": [None, 8, 12, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

dt_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    dt_param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
dt_grid.fit(X_train, y_train)
print("Best Decision Tree params:", dt_grid.best_params_)

y_pred_dt_tuned = dt_grid.best_estimator_.predict(X_test)
mse_dt_tuned, rmse_dt_tuned, r2_dt_tuned = print_regression_metrics(
    "Decision Tree Regressor (GridSearchCV)", y_test, y_pred_dt_tuned
)


In [ ]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 12, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)
print("Best Random Forest params (grid):", rf_grid.best_params_)

y_pred_rf_tuned = rf_grid.best_estimator_.predict(X_test)
mse_rf_tuned, rmse_rf_tuned, r2_rf_tuned = print_regression_metrics(
    "Random Forest Regressor (GridSearchCV)", y_test, y_pred_rf_tuned
)


In [ ]:
# RandomizedSearchCV draws max_depth from a distribution during search.
# Do not pre-sample a fixed list with .rvs() — that would freeze three depths
# at notebook parse time instead of searching them.
rf_param_distributions = {
    "n_estimators": randint(80, 220),
    "max_depth": randint(8, 30),
    "min_samples_split": randint(2, 11),
    "min_samples_leaf": randint(1, 5),
}

rf_random = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_param_distributions,
    n_iter=15,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=42,
)
rf_random.fit(X_train, y_train)
print("Best Random Forest params (random):", rf_random.best_params_)

y_pred_rf_random = rf_random.best_estimator_.predict(X_test)
mse_rf_random, rmse_rf_random, r2_rf_random = print_regression_metrics(
    "Random Forest Regressor (RandomizedSearchCV)", y_test, y_pred_rf_random
)


## 10. Regression comparison

Metrics are computed from the models above, not typed in by hand.


In [ ]:
results = pd.DataFrame(
    [
        ["Linear Regression", mse_lr, rmse_lr, r2_lr],
        ["Decision Tree (base)", mse_dt, rmse_dt, r2_dt],
        ["Decision Tree (tuned)", mse_dt_tuned, rmse_dt_tuned, r2_dt_tuned],
        ["Random Forest (base)", mse_rf, rmse_rf, r2_rf],
        ["Random Forest (GridSearchCV)", mse_rf_tuned, rmse_rf_tuned, r2_rf_tuned],
        [
            "Random Forest (RandomizedSearchCV)",
            mse_rf_random,
            rmse_rf_random,
            r2_rf_random,
        ],
    ],
    columns=["Model", "MSE", "RMSE", "R2"],
).sort_values("RMSE")

print(results.to_string(index=False, float_format=lambda x: f"{x:,.4f}" if x < 2 else f"{x:,.2f}"))

best_row = results.iloc[0]
print(
    f"\nLowest RMSE: {best_row['Model']} "
    f"(RMSE={best_row['RMSE']:,.2f}, R2={best_row['R2']:.4f})"
)


## 11. Optional: Random Forest Classifier (RFC)

To include RFC from the assignment brief, `price` is cut into three quantile bands. Features stay the same; **accuracy** is appropriate here because the target is now a class, not a dollar amount.


In [ ]:
df_hp["price_band"] = pd.qcut(df_hp["price"], q=3, labels=["Low", "Medium", "High"])
print(df_hp["price_band"].value_counts().sort_index())

X_cls = df_hp[feature_cols]
y_cls = df_hp["price_band"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train_c, y_train_c)
y_pred_dt_clf = dt_clf.predict(X_test_c)
acc_dt_clf = accuracy_score(y_test_c, y_pred_dt_clf)

rfc = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfc.fit(X_train_c, y_train_c)
y_pred_rfc = rfc.predict(X_test_c)
acc_rfc = accuracy_score(y_test_c, y_pred_rfc)

print(f"\nDecision Tree Classifier accuracy: {acc_dt_clf:.4f}")
print(f"Random Forest Classifier accuracy: {acc_rfc:.4f}")
print("\nRFC classification report:")
print(classification_report(y_test_c, y_pred_rfc))

plt.figure(figsize=(6, 5))
sns.heatmap(
    confusion_matrix(y_test_c, y_pred_rfc, labels=["Low", "Medium", "High"]),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Low", "Medium", "High"],
    yticklabels=["Low", "Medium", "High"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("RFC Confusion Matrix (price bands)")
plt.show()


## How to read the results

1. **Regression** is the main task: predict `price`. Compare models with **RMSE** and **R2**, not classification accuracy.
2. A default Decision Tree often overfits; GridSearchCV usually lowers RMSE.
3. Linear Regression is a strong baseline on this mostly linear-looking synthetic data.
4. RFC accuracy is a **different problem** (price band), so do not rank it in the same table as RMSE.
